<a href="https://colab.research.google.com/github/msiplab/VieWork/blob/master/code/python/vie_sec14_recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> <a href="https://matlab.mathworks.com/open/github/v1?repo=msiplab/VieWork&file=code/matlab/vie_sec14_recognition.m&project=VieWork.prj" target="_parent"><img src="https://www.mathworks.com/images/responsive/global/open-in-matlab-online.svg" alt="Open in MATLAB Online"/></a>

# 第14回 パターン認識と特徴抽出
村松正吾　「画像情報工学」

動作確認：Python 3.12（Google Colab），NumPy，SciPy，scikit-image，Matplotlib

MATLAB のライブスクリプト `code/matlab/vie_sec14_recognition.m` を Python に移植したもの。第14回のスライド（vie2026-14）で使う図と数値を計算する。特徴空間と最近傍決定則，色の特徴空間，ハリスのコーナー検出，キャニーの輪郭線検出（教科書の例題「プレウィットフィルタ」「勾配の大きさ」），ハフ変換による直線検出，活性化関数，畳み込み層とプーリング層を扱う。記号は教科書に合わせる。

スライドの図と数値は MATLAB 版で作っている。新潟大学の学生は[大学の包括ライセンス](https://jp.mathworks.com/academia/tah-portal/niigata-university-31576302.html)で MATLAB を使えるので，MATLAB 版もぜひ動かしてほしい。上の「Open in MATLAB Online」のボタンを押すと，ブラウザだけで MATLAB Online が開き，VieWork が自分の MATLAB Drive に複製されて，このノートブックに対応するライブスクリプトが開く（インストールは不要）。さらに [MATLAB Connector](https://jp.mathworks.com/products/matlab-drive.html) を手元の PC に入れると，MATLAB Drive のファイルが PC のフォルダと自動で同期されるので，MATLAB Online で作業したファイルを手元の MATLAB でもそのまま開ける。

画像処理には scikit-image，SciPy，NumPy を使う。教科書の例題の数値はスライドと一致することを `assert` で確かめている。画像から計算する値は，ライブラリの実装が MATLAB と異なるため，スライドの値とわずかに異なることがある。

## 準備
共通ヘルパー `vie.py`（サンプル画像の取得，日本語フォントと配色の設定）を読み込む。Colab では GitHub から取得する。

In [ ]:
import importlib
import os
import sys
import urllib.request

# Colab では毎回 GitHub から最新の vie.py を取得する（VieWork を clone した場合は同じフォルダの vie.py を使う）
if "google.colab" in sys.modules or not os.path.exists("vie.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/msiplab/VieWork/master/code/python/vie.py", "vie.py")

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
from scipy import ndimage
from skimage import feature, filters, transform
from skimage.util import img_as_float
import vie

vie = importlib.reload(vie)   # 取得し直した vie.py を読み込み直す

# 図の配色はスライドと揃え，ロゴの 3 色（メインの緑，寒色系の青，暖色系の橙）と灰色を使う
cmain, ccool, cwarm, cgray = vie.CMAIN, vie.CCOOL, vie.CWARM, vie.CGRAY


def show(ax, X, title=None):
    """画像を軸なし・縦横比 1 で表示する（グレースケールは 0 を黒，最大値（255 か 1）を白）．"""
    vmax = 255 if X.dtype == np.uint8 else 1
    ax.imshow(X, cmap="gray", vmin=0, vmax=vmax, interpolation="nearest")
    ax.set_axis_off()
    if title is not None:
        ax.set_title(title)

写真は教科書のサンプル画像 msipimg##.tif（512×512 のカラー）を `vie.msipimg` で読み込む。色の特徴空間には色とりどりの花束（msipimg02），コーナー・輪郭線の検出と特徴マップには縦横の縁がはっきりした石造りの建物（msipimg04），ハフ変換には斜めの直線（縞）が並ぶ路面（msipimg06），輪郭線の小さな例には丸い輪郭が並ぶマカロン（msipimg03）を使う。

図の配色はロゴの 3 色（メインの緑 #008855，青 #2E75B6，橙 #C55A11）と灰色を基本にする。薄い色は白と混ぜて作る。

In [ ]:
cLogo = np.array([cmain, ccool, cwarm])
cLight = 1 - 0.18 * (1 - cLogo)                  # 塗り用の薄い色

## 特徴空間とパターン学習
2 次元の特徴ベクトル $\mathbf{x}=(x_0\ \ x_1)^\top$ をもつ 3 クラス $\omega_1,\omega_2,\omega_3$ の学習データ（各 30 個）を作る。同じクラスのパターンは特徴空間上で塊（クラスタ）をなす。クラスごとの平均を代表ベクトル $\boldsymbol{\mu}_i$ とする（パターン学習）。

正規乱数は NumPy と MATLAB で作り方が違い，同じシードでも値が一致しない。スライドと同じ例にするため，MATLAB の `rng(0); randn(360,1)` の値を VieWork の `data/vie-14-randn.txt` に保存しておき，それを順に使う（学習データ，テストデータの順）。

In [ ]:
z = np.loadtxt(vie.datafile("vie-14-randn.txt"), comments="%", encoding="utf-8")   # MATLAB の randn の値
mu0 = np.array([[1, 1], [4, 1.5], [2.5, 4]])     # 真の中心
nTrain = 30                                      # クラスあたりの学習データ数
pos = 0                                          # 次に使う乱数の位置


def gendata():
    """各クラスの中心のまわりに標準偏差 0.6 の正規乱数で点を作る（クラスは 1, 2, 3）．

    MATLAB の randn(nTrain,2) と同じく，乱数を列優先で並べる．
    """
    global pos
    blocks = []
    for i in range(3):
        r = z[pos:pos + 2 * nTrain].reshape((nTrain, 2), order="F")
        pos += 2 * nTrain
        blocks.append(mu0[i] + 0.6 * r)
    return np.vstack(blocks), np.repeat([1, 2, 3], nTrain)


Xtr, ytr = gendata()
mu = np.array([Xtr[ytr == i].mean(axis=0) for i in (1, 2, 3)])
print("代表ベクトル（行が μ_i^T）\n", np.round(mu, 2))
# スライドの値と一致することを確かめる
assert np.allclose(np.round(mu, 2), [[1.33, 0.93], [4.03, 1.42], [2.38, 3.95]], rtol=0, atol=1e-9)

特徴空間上の学習データ（クラスごとに色分け）と代表ベクトル（＋印）。

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4.5), layout="constrained")
for i in range(3):
    ax.scatter(*Xtr[ytr == i + 1].T, 20, color=cLogo[i], label=rf"$\omega_{i + 1}$")
for i in range(3):
    ax.plot(*mu[i], "k+", ms=14, mew=2.5)
    ax.text(mu[i, 0] + 0.2, mu[i, 1] + 0.35, rf"$\boldsymbol{{\mu}}_{i + 1}$", fontsize=14,
            bbox=dict(fc="w", ec="none", pad=0.5))
ax.set(xlim=(-1, 6), ylim=(-1, 6), aspect="equal", xlabel="$x_0$", ylabel="$x_1$")
ax.grid(True)
ax.legend(loc="upper right")
plt.show()

## パターン識別：最近傍決定則
未知の入力 $\mathbf{x}$ は，最も近い代表ベクトルのクラスに識別する（最近傍決定則）。距離はユークリッド距離 $\|\mathbf{x}-\boldsymbol{\mu}_i\|_2$ で測る。

In [ ]:
xq = np.array([3, 2.5])                          # 未知入力
d = np.linalg.norm(mu - xq, axis=1)              # 各代表ベクトルまでの距離
cls = int(np.argmin(d)) + 1                      # クラス番号は 1, 2, 3
print("距離:", ", ".join(f"{v:.2f}" for v in d), " → クラス", cls)
assert cls == 2 and np.allclose(np.round(d, 2), [2.29, 1.50, 1.58], rtol=0, atol=1e-9)

前回の演習課題（14）の図にならい，未知入力 $\mathbf{x}$（☆印）から各代表ベクトルへ線分を引き，距離を添える。背景は各点を最も近い代表ベクトルのクラスで塗り分けたもの（識別境界は代表ベクトルどうしの垂直二等分線になる）。

In [ ]:
def nearest(Xq, mu):
    """各行の点に最も近い代表ベクトルのクラス番号（1 始まり）．"""
    return np.argmin(np.linalg.norm(Xq[:, None, :] - mu[None], axis=2), axis=1) + 1


g = np.linspace(0, 5, 301)
g1, g2 = np.meshgrid(g, g)
lab = nearest(np.column_stack([g1.ravel(), g2.ravel()]), mu).reshape(g1.shape)
fig, ax = plt.subplots(figsize=(4.5, 4.5), layout="constrained")
ax.imshow(lab, cmap=ListedColormap(cLight), vmin=0.5, vmax=3.5, origin="lower", extent=(0, 5, 0, 5))
for i in range(3):
    ax.scatter(*Xtr[ytr == i + 1].T, 12, color=cLogo[i], alpha=0.35)
for i in range(3):
    ls, lw = ("-", 3) if i + 1 == cls else ("--", 1.5)
    ax.plot([xq[0], mu[i, 0]], [xq[1], mu[i, 1]], ls, color=cLogo[i], lw=lw)
    ax.text((xq[0] + mu[i, 0]) / 2, (xq[1] + mu[i, 1]) / 2, f"{d[i]:.2f}", fontsize=12, ha="center",
            va="center", color=cLogo[i], bbox=dict(fc="w", ec="none", pad=1))
    ax.plot(*mu[i], "k+", ms=14, mew=2.5)
    ax.text(mu[i, 0] + 0.15, mu[i, 1] - 0.3, rf"$\boldsymbol{{\mu}}_{i + 1}$", fontsize=14,
            bbox=dict(fc="w", ec="none", pad=0.5))
ax.plot(*xq, "k*", ms=18, mfc="w", mew=1.5)
ax.text(xq[0] + 0.2, xq[1] + 0.2, r"$\mathbf{x}$", fontsize=14)
ax.set(xlim=(0, 5), ylim=(0, 5), aspect="equal", xlabel="$x_0$", ylabel="$x_1$")
plt.show()

新しいテストデータ（各クラス 30 個）での識別率。

In [ ]:
Xte, yte = gendata()
yhat = nearest(Xte, mu)
acc = np.mean(yhat == yte)
print(f"識別率 {100 * acc:.0f} %（テストデータ {yte.size} 個）")

## 色の特徴空間
画像の各画素の (R, G, B) を特徴ベクトルとみなすと，RGB 空間が特徴空間になる（色そのものが内容なので，点は画素の色で塗る）。色の分布が広がって見えるよう，橙・黄・赤・白・緑の花が並ぶ花束（msipimg02）を使う。

In [ ]:
Xc = img_as_float(vie.msipimg(2))
Xs = transform.rescale(Xc, 1 / 8, order=3, anti_aliasing=True, channel_axis=2)
v = np.clip(Xs.reshape(-1, 3), 0, 1)
fig = plt.figure(figsize=(9, 4), layout="constrained")
ax0 = fig.add_subplot(1, 2, 1)
show(ax0, np.clip(transform.rescale(Xc, 0.25, order=3, anti_aliasing=True, channel_axis=2), 0, 1))
ax = fig.add_subplot(1, 2, 2, projection="3d")
ax.scatter(v[:, 0], v[:, 1], v[:, 2], s=8, c=v, depthshade=False)
ax.set(xlim=(0, 1), ylim=(0, 1), zlim=(0, 1), xlabel="R", ylabel="G", zlabel="B")
ax.view_init(elev=25, azim=-130)                 # 視点（仰角 25°，方位角 −130°）
plt.show()

## ハリスのコーナー検出：局所構造行列の数値例
構造テンソル $\hat{\mathbf{M}}=\begin{pmatrix}\hat{A}&\hat{C}\\\hat{C}&\hat{B}\end{pmatrix}$，$A=x_\mathrm{v}^2,\ B=x_\mathrm{h}^2,\ C=x_\mathrm{v}x_\mathrm{h}$（近傍で平均）と，コーナー応答値 $r=\det(\hat{\mathbf{M}})-k(\mathrm{trace}(\hat{\mathbf{M}}))^2$（$k=0.04$）を，平坦・エッジ・コーナーの $5\times5$ パッチで比べる。

In [ ]:
k = 0.04
corner_patch = np.zeros((5, 5))
corner_patch[2:, 2:] = 1
P = [np.zeros((5, 5)), np.hstack([np.zeros((5, 2)), np.ones((5, 3))]), corner_patch]
names = ["平坦", "エッジ", "コーナー"]
fv = np.array([[-1], [0], [1]])
fh = np.array([[-1, 0, 1]])                      # 中心差分
R = np.zeros((3, 3))                             # 列：det, trace, r
for i in range(3):
    xv = ndimage.correlate(P[i], fv, mode="nearest")
    xh = ndimage.correlate(P[i], fh, mode="nearest")
    Ah, Bh, Ch = np.mean(xv**2), np.mean(xh**2), np.mean(xv * xh)
    M = np.array([[Ah, Ch], [Ch, Bh]])
    R[i] = [np.linalg.det(M), np.trace(M), np.linalg.det(M) - k * np.trace(M) ** 2]
print(f"{'':6s} {'det':>7s} {'trace':>7s} {'r':>7s}")
for nm, (dt, tr, r) in zip(names, R):
    print(f"{nm:5s} {dt:7.3f} {tr:7.2f} {r:7.3f}")
assert [f"{a:.3f}" for a in R[:, 2]] == ["0.000", "-0.006", "0.047"]   # スライドの値と一致
assert [f"{a:.3f}" for a in R[:, 0]] == ["0.000", "0.000", "0.056"]
assert [f"{a:.2f}" for a in R[:, 1]] == ["0.00", "0.40", "0.48"]

## ハリスのコーナー検出：処理結果
市松模様と石造りの建物の画像（msipimg04 をグレースケールにして 256×256 に縮小）でコーナーを検出する。建物はアーチの頂や柱の角など，縦横の縁が交わる点がはっきりしている。コーナー点は橙の＋印で示す。

コーナー応答値は scikit-image の `feature.corner_harris`（$k=0.04$）で求め，その極大点を `feature.corner_peaks` で応答値の強い順に取り出す（最大値の 1% 以上，市松模様は 40 個まで，建物は 150 個まで）。

In [ ]:
Cb = (np.add.outer(np.arange(144) // 24, np.arange(144) // 24) % 2).astype(float)   # 24 画素四方の白黒のます目を 6×6 に並べた市松模様
cc = feature.corner_peaks(feature.corner_harris(Cb, k=0.04), min_distance=3, threshold_rel=0.01, num_peaks=40)
Gr = img_as_float(vie.msipimg(4, 256, "gray"))
cg = feature.corner_peaks(feature.corner_harris(Gr, k=0.04), min_distance=1, threshold_rel=0.01, num_peaks=150)
print("コーナー点の数：市松模様", len(cc), "，建物", len(cg))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9, 4.4), layout="constrained", width_ratios=[1, 1])
show(axs[0], Cb, "市松模様")
axs[0].plot(cc[:, 1], cc[:, 0], "+", color=cwarm, ms=10, mew=2.5)
show(axs[1], Gr, "石造りの建物")
axs[1].plot(cg[:, 1], cg[:, 0], "+", color=cwarm, ms=7, mew=2)
plt.show()

## キャニーの輪郭線検出
ガウス平滑化 → 勾配 → 非極大抑制 → ヒステリシス閾値処理。比較としてソーベル勾配の単純な閾値処理も示す。

キャニーの輪郭線検出には scikit-image の `feature.canny`（ガウス平滑化の標準偏差 $\sqrt{2}$，閾値は既定値）を使う。ソーベルの方は `filters.sobel` で求めた勾配の大きさを，大津法の閾値（`filters.threshold_otsu`）で二値化する。

In [ ]:
Ec = feature.canny(Gr, sigma=np.sqrt(2))
mag = filters.sobel(Gr)                          # ソーベル勾配の大きさ
Es = mag > filters.threshold_otsu(mag)
Xk = img_as_float(vie.msipimg(3, 256, "gray"))   # マカロン（丸い輪郭が並ぶ）
Ek = feature.canny(Xk, sigma=np.sqrt(2))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8, 4.2), layout="constrained")
show(axs[0], (~Ec).astype(float), "キャニー")
show(axs[1], (~Es).astype(float), "ソーベル")
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8, 4.2), layout="constrained")
show(axs[0], Xk, "マカロン")
show(axs[1], (~Ek).astype(float), "キャニー")
plt.show()

## 教科書の例題「プレウィットフィルタ」「勾配の大きさ」（3.2 節）
キャニーの輪郭線検出の勾配計算の例。教科書の例題「矩形フィルタ」の配列 $\mathsf{x}$ にプレウィットフィルタ（垂直方向 $f_\mathrm{v}$，水平方向 $f_\mathrm{h}$）を施し，勾配ベクトル $\vec{y}[\boldsymbol{n}]=(x_\mathrm{v}[\boldsymbol{n}]\ \ x_\mathrm{h}[\boldsymbol{n}])^\top$ の大きさ $y_\mathrm{mag}[\boldsymbol{n}]=\sqrt{x_\mathrm{v}^2[\boldsymbol{n}]+x_\mathrm{h}^2[\boldsymbol{n}]}$ を求める。周囲の値はすべて零値とする。

教科書の線形フィルタ（式 (3.1)）は局所的な積和（相関）の形なので，`scipy.ndimage.correlate`（零値拡張は `mode="constant"`）がそのまま使える。

In [ ]:
xp = np.array([[18, 9, 9, 9], [27, 9, 9, 9], [36, 9, 9, 9]])   # 教科書の例題「矩形フィルタ」の配列
fvP = np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]])          # プレウィットフィルタ（垂直方向）
fhP = fvP.T                                                    # プレウィットフィルタ（水平方向）
Xv = ndimage.correlate(xp, fvP, mode="constant")              # 垂直方向の差分 x_v
Xh = ndimage.correlate(xp, fhP, mode="constant")              # 水平方向の差分 x_h
Ymag = np.sqrt(Xv**2 + Xh**2)                                 # 勾配の大きさ y_mag
print("x_v =\n", Xv)
print("x_h =\n", Xh)
print("y_mag =\n", np.round(Ymag, 2))

教科書の解答と一致することを確かめる。

In [ ]:
XvBook = [[36, 45, 27, 18], [18, 18, 0, 0], [-36, -45, -27, -18]]
XhBook = [[18, -27, 0, -18], [27, -54, 0, -27], [18, -45, 0, -18]]
YmagBook = [[40.25, 52.48, 27.00, 25.46], [32.45, 56.92, 0.00, 27.00], [40.25, 63.64, 27.00, 25.46]]
assert (np.array_equal(Xv, XvBook) and np.array_equal(Xh, XhBook)
        and np.allclose(np.round(Ymag, 2), YmagBook, rtol=0, atol=1e-9)), "教科書の例題の解答と一致しない"

## ハフ変換：数値例
エッジ画素の位置 $\boldsymbol{q}=(q_1\ \ q_2)^\top$ を $\rho=q_1\cos\theta+q_2\sin\theta$ で $(\theta,\rho)$ 空間の曲線に写す。一直線上の 3 点 $\boldsymbol{q}_\mathrm{A}=(0\ \ 2)^\top$，$\boldsymbol{q}_\mathrm{B}=(1\ \ 1)^\top$，$\boldsymbol{q}_\mathrm{C}=(2\ \ 0)^\top$ は，$\theta=45^\circ$ でいずれも $\rho=\sqrt{2}$ となる（3 本の曲線が 1 点で交わる）。

In [ ]:
pts = np.array([[0, 2], [1, 1], [2, 0]])
th = np.deg2rad(45)
rho = pts[:, 0] * np.cos(th) + pts[:, 1] * np.sin(th)
print("ρ =", np.round(rho, 3))
assert np.allclose(rho, np.sqrt(2))

In [ ]:
tt = np.linspace(0, np.pi, 361)
fig, ax = plt.subplots(figsize=(5.5, 3.6), layout="constrained")
for i, nm in enumerate("ABC"):
    ax.plot(np.rad2deg(tt), pts[i, 0] * np.cos(tt) + pts[i, 1] * np.sin(tt), color=cLogo[i], lw=2,
            label=rf"$\boldsymbol{{q}}_\mathrm{{{nm}}}=({pts[i, 0]}\ {pts[i, 1]})^\top$")
ax.plot(45, np.sqrt(2), "ko", ms=10, mfc="none", mew=2)
ax.set(xlim=(0, 180), xticks=range(0, 181, 45), xlabel=r"$\theta$ [度]", ylabel=r"$\rho$")
ax.grid(True)
ax.legend(loc="lower left")
plt.show()

## ハフ変換：処理例
斜めの直線（縞の縁）がはっきりした路面の画像（msipimg06 をグレースケールにして 256×256 に縮小）を使う。路面の細かな模様の輪郭を拾わないよう，キャニーのガウス平滑化の標準偏差を 3 と大きめにとる。

キャニーの輪郭画像を投票し，投票度数の大きなセル（ピーク）を探索して直線を描く（逆ハフ変換）。投票度数は見やすいよう平方根をとって白→緑→濃緑の色で表示する。選ばれたセルは橙の□印。

投票は scikit-image の `transform.hough_line`（$\theta$ は $-90^\circ$ から $89^\circ$ まで $1^\circ$ 刻み，画素の位置は左上を原点とする $(q_1,q_2)=(n_\mathrm{h},n_\mathrm{v})$），ピークの探索は `transform.hough_line_peaks`（最大 12 個，最大値の 30% 以上）で行う。画像に描く線分は `transform.probabilistic_hough_line`（投票の閾値 50，線分の長さ 60 画素以上，隙間 20 画素以下はつなぐ）で取り出す。

In [ ]:
Gh = img_as_float(vie.msipimg(6, 256, "gray"))
Eh = feature.canny(Gh, sigma=3)
T = np.arange(-90, 90)                           # θ [度]
Hh, angles, Rr = transform.hough_line(Eh, theta=np.deg2rad(T))
_, pkA, pkD = transform.hough_line_peaks(Hh, angles, Rr, num_peaks=12, threshold=0.3 * Hh.max())
hl = transform.probabilistic_hough_line(Eh, threshold=50, line_length=60, line_gap=20, theta=np.deg2rad(T), rng=0)
cmapV = LinearSegmentedColormap.from_list("wg", [(1, 1, 1), cmain, (0, 0.2, 0.12)])   # 白→緑→濃緑

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8, 4.2), layout="constrained")
show(axs[0], Gh, "路面")
show(axs[1], (~Eh).astype(float), r"キャニー（$\sigma=3$）")
plt.show()

$\theta\in[-90^\circ,90^\circ)$ で投票した。スライドの $\theta\in[0,\pi)$，$\rho\in\mathbb{R}$ に合わせ，$\theta<0$ の列を $(\theta+180^\circ,-\rho)$ に移して表示する（同じ直線の表現）。

In [ ]:
assert np.array_equal(Rr, -Rr[::-1])             # rho の刻みは原点に対称
neg = T < 0
H180 = np.hstack([Hh[:, ~neg], Hh[::-1, neg]])   # theta in [0,90) と，[-90,0) を反転した [90,180)
T180 = np.concatenate([T[~neg], T[neg] + 180])
pkT, pkR = np.rad2deg(pkA).round(), pkD.copy()
pkR[pkT < 0] = -pkR[pkT < 0]
pkT[pkT < 0] += 180
Hs = (H180 - H180.min()) / (H180.max() - H180.min())
fig, ax = plt.subplots(figsize=(6, 4.2), layout="constrained")
ax.imshow(np.sqrt(Hs), cmap=cmapV, aspect="auto", interpolation="nearest",
          extent=(T180[0] - 0.5, T180[-1] + 0.5, Rr[-1] + 0.5, Rr[0] - 0.5))   # 投票度数（平方根）
ax.plot(pkT, pkR, "s", color=cwarm, ms=7, mfc="none", mew=2)
ax.set(xticks=range(0, 181, 45), xlabel=r"$\theta$ [度]", ylabel=r"$\rho$")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4.5), layout="constrained")
show(ax, Gh)
for p1, p2 in hl:                                # 端点は (列, 行)
    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], lw=3, color=cmain)
plt.show()
print("線分の数:", len(hl))

## 活性化関数
活性化関数 $\phi(\cdot)$ の例（参考資料 2 章）：シグモイド関数 $1/(1+\mathrm{e}^{-au})$（$a=1$，章末問題），ReLU（整流線形関数） $\max(0,u)$（2.1.5 項，ヒンジ関数で $\lambda=0$ としたランプ関数）。

In [ ]:
u = np.linspace(-4, 4, 401)
fig, ax = plt.subplots(figsize=(5, 3.6), layout="constrained")
ax.plot(u, 1 / (1 + np.exp(-u)), color=cmain, lw=2, label=r"シグモイド $1/(1+\mathrm{e}^{-u})$")
ax.plot(u, np.maximum(u, 0), color=cwarm, lw=2, label=r"ReLU $\max(0,u)$")
ax.set(ylim=(-0.5, 2), xlabel="$u$", ylabel=r"$\phi(u)$")
ax.grid(True)
ax.legend(loc="upper left")
plt.show()

## 畳み込み層とプーリング層
4 種類のフィルタ（フィルタバンク）で畳み込み，ReLU を施し，$2\times2$ の最大値プーリングで縮小した特徴マップを並べる。入力は石造りの建物（msipimg04 をグレースケールにして 256×256 に縮小）で，柱の縦の縁と階段の横の縁がそれぞれ別の特徴マップに現れ，フィルタの方向選択性が分かる。

In [ ]:
Xg = img_as_float(vie.msipimg(4, 256, "gray"))
# 水平差分（Sobel），垂直差分（Sobel），ラプラシアン，符号反転したラプラシアン
lap = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]])
Fk = [np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]), np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]), lap, -lap]
maps = []
for F in Fk:
    Z = np.maximum(ndimage.correlate(Xg, F, mode="nearest"), 0)            # 畳み込み＋ReLU
    Zp = Z.reshape(Z.shape[0] // 2, 2, Z.shape[1] // 2, 2).max(axis=(1, 3))   # 最大値プーリング
    maps.append(Zp / Zp.max())

In [ ]:
fig, axs = plt.subplots(1, 5, figsize=(13, 2.9), layout="constrained")
show(axs[0], Xg, "入力")
for ax, Mp, ttl in zip(axs[1:], maps, ("水平差分", "垂直差分", "ラプラシアン", "−ラプラシアン")):
    show(ax, Mp, ttl)
plt.show()

## まとめ
- パターン認識は特徴抽出と識別（識別辞書との照合）からなる。最近傍決定則は最も近い代表ベクトルのクラスを出力する
- ハリスのコーナー検出，キャニーの輪郭線検出，ハフ変換はガウシアンフィルタと勾配フィルタ（第5回）を土台にする
- ニューラルネットワークは重み付け和と活性化関数の多層構造で，損失関数（1/2 倍した平均二乗誤差）の勾配を誤差逆伝播法で求めて学習する。畳み込み層はフィルタバンクに相当する

© Copyright, Shogo MURAMATSU, All rights reserved.